# Capstone: End-to-End Deep Learning Project
## Handwritten Digit Classifier — Full PyTorch Production Pipeline

This capstone builds a **complete, production-quality deep learning pipeline** in PyTorch — from raw data through training, evaluation, debugging, and deployment.

### What You Will Learn
- PyTorch Dataset and DataLoader patterns
- Building CNN architectures with best practices
- Training loop with validation, early stopping, LR scheduling
- Monitoring: loss curves, gradient norms
- Debugging: common training failures and how to fix them
- Data augmentation for regularization
- Model checkpointing and resuming
- Evaluation: accuracy, confusion matrix, per-class analysis
- TorchScript export for production
- REST API deployment with FastAPI

### Architecture Overview
```
Input Image (28×28×1)
  → Conv2d(1→32, 3×3) + ReLU + BatchNorm
  → MaxPool2d(2×2)             [14×14×32]
  → Conv2d(32→64, 3×3) + ReLU + BatchNorm
  → MaxPool2d(2×2)             [7×7×64]
  → Flatten → Dropout(0.5)
  → Linear(3136→256) + ReLU
  → Linear(256→10)
  → Softmax → class (0-9)
```

## Real-World Analogy

Training a deep learning model is like **teaching a baby to recognise faces**:
- **Random initialisation** = a newborn — sees nothing meaningful yet
- **Forward pass** = the baby looks at a face and guesses
- **Loss** = the parent's correction: "That's not mum, that's dad!"
- **Backward pass** = the brain adjusts its connections based on the correction
- **Epochs** = years of experience — seeing millions of faces
- **BatchNorm** = making sure the baby is equally good at faces in bright and dim light
- **Early stopping** = knowing when the baby has learned enough (before memorising one specific face)


## Step 1: Setup

In [ ]:
import numpy as np
import time
import os
import tempfile
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, random_split
    from torch.optim.lr_scheduler import OneCycleLR, ReduceLROnPlateau
    TORCH_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed — simulated output shown.")
    print("Install: pip install torch torchvision")

# Try to load MNIST (requires torchvision)
try:
    from torchvision import datasets, transforms
    TORCHVISION_AVAILABLE = True
    print(f"torchvision available")
except ImportError:
    TORCHVISION_AVAILABLE = False
    print("torchvision not available — will use synthetic data")

## Step 2: Data Loading and Augmentation

In [ ]:
if TORCH_AVAILABLE:
    if TORCHVISION_AVAILABLE:
        # Real MNIST dataset
        train_transforms = transforms.Compose([
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean/std
        ])
        test_transforms = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])

        data_dir = tempfile.mkdtemp()
        train_dataset = datasets.MNIST(data_dir, train=True,  download=True, transform=train_transforms)
        test_dataset  = datasets.MNIST(data_dir, train=False, download=True, transform=test_transforms)

        # Split train into train + validation
        n_val   = 5000
        n_train = len(train_dataset) - n_val
        train_ds, val_ds = random_split(train_dataset, [n_train, n_val],
                                         generator=torch.Generator().manual_seed(42))
        print(f"MNIST loaded: train={n_train}, val={n_val}, test={len(test_dataset)}")
        IMG_CHANNELS = 1
        NUM_CLASSES  = 10

    else:
        # Synthetic grayscale image dataset
        class SyntheticDigitDataset(Dataset):
            def __init__(self, n=10000):
                torch.manual_seed(42)
                self.images = torch.randn(n, 1, 28, 28)
                self.labels = torch.randint(0, 10, (n,))
            def __len__(self): return len(self.labels)
            def __getitem__(self, idx): return self.images[idx], self.labels[idx]

        full_ds = SyntheticDigitDataset(10000)
        n_train, n_val, n_test = 7000, 1000, 2000
        train_ds, val_ds, test_dataset = random_split(full_ds, [n_train, n_val, n_test])
        print(f"Synthetic dataset: train={n_train}, val={n_val}, test={n_test}")
        IMG_CHANNELS = 1
        NUM_CLASSES  = 10

    # DataLoaders
    BATCH_SIZE = 64
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Batches per epoch: {len(train_loader)}")

    # Inspect a batch
    images, labels = next(iter(train_loader))
    print(f"Batch shape: images={images.shape}, labels={labels.shape}")
    print(f"Pixel range: [{images.min():.2f}, {images.max():.2f}]")

else:
    print("Data loading (simulated):")
    print()
    print("  # With torchvision:")
    print("  train_transforms = transforms.Compose([")
    print("      transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),")
    print("      transforms.ToTensor(),")
    print("      transforms.Normalize((0.1307,), (0.3081,)),")
    print("  ])")
    print("  MNIST: 60,000 train, 10,000 test, 28×28 grayscale images")
    print("  After split: 55,000 train / 5,000 val / 10,000 test")
    print("  Batch shape: [64, 1, 28, 28]")

## Step 3: CNN Model Architecture

In [ ]:
if TORCH_AVAILABLE:
    class DigitCNN(nn.Module):
        """Convolutional neural network for digit classification."""

        def __init__(self, num_classes=10, dropout=0.5):
            super().__init__()

            # Convolutional blocks
            self.block1 = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 28×28 → 28×28
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),                               # 28×28 → 14×14
                nn.Dropout2d(0.1),
            )

            self.block2 = nn.Sequential(
                nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 14×14 → 14×14
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.Conv2d(64, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),                               # 14×14 → 7×7
                nn.Dropout2d(0.2),
            )

            # Fully connected layers
            self.classifier = nn.Sequential(
                nn.Flatten(),                                   # 7×7×64 = 3136
                nn.Linear(7 * 7 * 64, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(256, num_classes),
            )

            # Weight initialization
            self._initialize_weights()

        def _initialize_weights(self):
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                    if m.bias is not None: nn.init.zeros_(m.bias)
                elif isinstance(m, nn.Linear):
                    nn.init.xavier_normal_(m.weight)
                    nn.init.zeros_(m.bias)

        def forward(self, x):
            x = self.block1(x)
            x = self.block2(x)
            return self.classifier(x)

    model = DigitCNN(num_classes=NUM_CLASSES).to(DEVICE)

    # Count parameters
    total_params    = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: DigitCNN")
    print(f"  Total parameters:     {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print()

    # Verify forward pass
    dummy = torch.randn(4, 1, 28, 28).to(DEVICE)
    out   = model(dummy)
    print(f"Forward pass: input {dummy.shape} → output {out.shape}")

else:
    print("CNN Architecture (simulated):")
    print()
    print("  class DigitCNN(nn.Module):")
    print("      block1: Conv2d(1→32) + BN + ReLU + Conv2d(32→32) + BN + ReLU + MaxPool + Dropout")
    print("      block2: Conv2d(32→64) + BN + ReLU + Conv2d(64→64) + BN + ReLU + MaxPool + Dropout")
    print("      classifier: Flatten → Linear(3136→256) + BN + ReLU + Dropout → Linear(256→10)")
    print()
    print("  Total parameters: 862,314")
    print("  Forward: [4,1,28,28] → [4,10]")

## Step 4: Training Loop with Validation, Checkpointing, Early Stopping

In [ ]:
if TORCH_AVAILABLE:
    def train_epoch(model, loader, optimizer, criterion, device, scheduler=None):
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)  # slightly faster than zero_grad()
            logits = model(images)
            loss   = criterion(logits, labels)
            loss.backward()
            # Gradient clipping: prevents exploding gradients
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            if scheduler: scheduler.step()
            total_loss += loss.item() * len(images)
            correct    += (logits.argmax(1) == labels).sum().item()
            n          += len(images)
        return total_loss / n, correct / n

    @torch.no_grad()
    def evaluate(model, loader, criterion, device):
        model.eval()
        total_loss, correct, n = 0.0, 0, 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * len(images)
            correct    += (logits.argmax(1) == labels).sum().item()
            n          += len(images)
        return total_loss / n, correct / n

    # Training config
    LR        = 1e-3
    N_EPOCHS  = 15
    PATIENCE  = 5       # early stopping patience
    ckpt_dir  = tempfile.mkdtemp()
    ckpt_path = os.path.join(ckpt_dir, 'best_model.pt')

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing prevents overconfidence
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = OneCycleLR(optimizer, max_lr=LR, epochs=N_EPOCHS, steps_per_epoch=len(train_loader))

    best_val_loss = float('inf')
    patience_count = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print(f"Training for up to {N_EPOCHS} epochs (patience={PATIENCE})...")
    print(f"{'Epoch':>5} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'LR':>10}")
    print("-" * 62)

    t_start = time.time()
    for epoch in range(1, N_EPOCHS + 1):
        t0 = time.time()

        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scheduler)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        current_lr = optimizer.param_groups[0]['lr']
        epoch_time = time.time() - t0

        flag = ''
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(),
                        'val_loss': val_loss, 'val_acc': val_acc}, ckpt_path)
            flag = ' ✓ saved'
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

        print(f"{epoch:5d} {train_loss:11.4f} {train_acc:10.4f} {val_loss:10.4f} {val_acc:9.4f} {current_lr:10.6f}{flag}")

    total_time = time.time() - t_start
    print(f"\nTraining complete in {total_time:.0f}s")

    # Load best checkpoint
    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt['model_state'])
    print(f"Best model: epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

else:
    print("Training loop (simulated):")
    print()
    print("  Epoch Train Loss  Train Acc  Val Loss  Val Acc")
    print("  ──────────────────────────────────────────────")
    print("      1    0.4821     0.8512    0.2134   0.9346  ✓ saved")
    print("      2    0.2301     0.9312    0.1456   0.9578  ✓ saved")
    print("      3    0.1823     0.9467    0.1187   0.9641  ✓ saved")
    print("      4    0.1512     0.9556    0.1034   0.9687  ✓ saved")
    print("      5    0.1289     0.9623    0.0987   0.9712  ✓ saved")
    print("      ...")
    print("     15    0.0876     0.9734    0.0812   0.9789  ✓ saved")
    print()
    print("  Best: epoch 15, val_acc=0.9789")

## Step 5: Final Evaluation on Test Set

In [ ]:
if TORCH_AVAILABLE:
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs  = F.softmax(logits, dim=1)
            preds  = logits.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    test_acc = (all_preds == all_labels).mean()
    print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

    # Per-class accuracy
    print("\nPer-class accuracy:")
    class_names = [str(i) for i in range(10)]
    for cls in range(10):
        mask = all_labels == cls
        acc  = (all_preds[mask] == all_labels[mask]).mean()
        n    = mask.sum()
        bar  = '█' * int(acc * 30)
        print(f"  Digit {cls}: {bar} {acc:.3f}  (n={n})")

    # Confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(all_labels, all_preds)
    print("\nConfusion matrix:")
    print("     " + " ".join(f"{i:4d}" for i in range(10)))
    for i, row in enumerate(cm):
        print(f"  {i}: " + " ".join(f"{v:4d}" for v in row))

    # Most confused pairs
    np.fill_diagonal(cm, 0)
    flat_idx = np.unravel_index(cm.argsort(axis=None)[::-1][:5], cm.shape)
    print("\nMost common confusions (true → predicted):")
    for t, p in zip(*flat_idx):
        print(f"  {t} → {p}: {cm[t, p]} times")

else:
    print("Test evaluation (simulated):")
    print()
    print("  Test Accuracy: 0.9923 (99.23%)")
    print()
    print("  Per-class accuracy:")
    accs = [0.994, 0.998, 0.991, 0.988, 0.993, 0.987, 0.995, 0.992, 0.985, 0.989]
    for i, acc in enumerate(accs):
        bar = '█' * int(acc * 30)
        print(f"  Digit {i}: {bar} {acc:.3f}")
    print()
    print("  Most common confusions:")
    print("  4 → 9: 7 times")
    print("  3 → 8: 5 times")
    print("  5 → 6: 4 times")

## Step 6: Export for Production — TorchScript

In [ ]:
if TORCH_AVAILABLE:
    # TorchScript: compile model to serializable format
    # No Python dependency needed at serving time
    model.eval()
    scripted = torch.jit.script(model)

    script_path = os.path.join(ckpt_dir, 'digit_cnn_scripted.pt')
    scripted.save(script_path)

    file_mb = os.path.getsize(script_path) / 1e6
    print(f"TorchScript saved: {script_path}")
    print(f"Size: {file_mb:.1f} MB")
    print()

    # Verify: load and predict
    loaded_script = torch.jit.load(script_path)
    test_img = torch.randn(1, 1, 28, 28)
    with torch.no_grad():
        logits = loaded_script(test_img)
        prob   = F.softmax(logits, dim=1)
        pred   = logits.argmax(1).item()

    print(f"TorchScript prediction: digit={pred}, confidence={prob.max().item():.3f}")
    print()
    print("TorchScript advantages:")
    print("  - No Python dependency: run in C++, Java, mobile apps")
    print("  - Faster inference: ahead-of-time compilation")
    print("  - Serializable: save to disk, version control")

else:
    print("TorchScript export (simulated):")
    print()
    print("  model.eval()")
    print("  scripted = torch.jit.script(model)")
    print("  scripted.save('digit_cnn_scripted.pt')")
    print()
    print("  # Load in any environment (no Python needed):")
    print("  loaded = torch.jit.load('digit_cnn_scripted.pt')")
    print("  pred = loaded(image_tensor).argmax(1).item()")
    print()
    print("  Other export options:")
    print("  - ONNX: torch.onnx.export(model, dummy_input, 'model.onnx')")
    print("    → Use with ONNX Runtime (fast inference in Python/C++/Java)")
    print("  - TensorRT: convert ONNX to TensorRT for NVIDIA GPU acceleration")
    print("  - CoreML: torch.jit → coremltools for iOS/macOS deployment")

## Step 7: Production API

In [ ]:
FASTAPI_CODE = '''
# app.py — run: uvicorn app:app --host 0.0.0.0 --port 8000
import torch
import torch.nn.functional as F
import numpy as np
from fastapi import FastAPI, File, UploadFile
from pydantic import BaseModel
from PIL import Image
import io, base64

# Load TorchScript model once at startup
MODEL   = torch.jit.load('digit_cnn_scripted.pt', map_location='cpu')
MODEL.eval()
MEAN, STD = 0.1307, 0.3081  # MNIST normalization constants

app = FastAPI(title="Digit Classifier API")

def preprocess(image: Image.Image) -> torch.Tensor:
    """Resize to 28×28, normalize, add batch+channel dims."""
    img = image.convert('L').resize((28, 28))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD
    return torch.tensor(arr).unsqueeze(0).unsqueeze(0)  # [1, 1, 28, 28]

class PredictionResponse(BaseModel):
    predicted_digit: int
    confidence: float
    all_probabilities: list[float]

@app.post("/predict", response_model=PredictionResponse)
async def predict(file: UploadFile = File(...)):
    """Upload a PNG/JPEG image of a handwritten digit."""
    contents = await file.read()
    img = Image.open(io.BytesIO(contents))
    tensor = preprocess(img)

    with torch.no_grad():
        logits = MODEL(tensor)
        probs  = F.softmax(logits, dim=1)[0]
        pred   = logits.argmax(1).item()

    return PredictionResponse(
        predicted_digit=pred,
        confidence=round(float(probs[pred]), 4),
        all_probabilities=[round(float(p), 4) for p in probs],
    )

@app.get("/health")
def health():
    return {"status": "ok", "model": "DigitCNN (TorchScript)"}
'''

print("Production API (app.py):")
print(FASTAPI_CODE)
print("Test with: curl -X POST http://localhost:8000/predict -F 'file=@digit.png'")

## Step 8: Common DL Debugging Checklist

In [ ]:
print("=" * 60)
print("DL TRAINING DEBUGGING CHECKLIST")
print("=" * 60)
print()

checks = [
    {
        'symptom':  'Loss is NaN from the start',
        'causes':   ['Learning rate too high', 'Input not normalized', 'Log(0) somewhere'],
        'fixes':    ['Lower LR by 10×', 'Normalize: (x - mean) / std', 'Add epsilon: log(x + 1e-8)'],
    },
    {
        'symptom':  'Loss decreases but accuracy stays at 10% (random)',
        'causes':   ['Labels are wrong (off by one, wrong mapping)', 'Model output not matching loss input'],
        'fixes':    ['Print first 5 labels and verify', 'Check CrossEntropyLoss: expects logits, not probabilities'],
    },
    {
        'symptom':  'Train loss low, val loss high (overfitting)',
        'causes':   ['Model too large', 'Not enough data', 'No regularization'],
        'fixes':    ['Reduce model size', 'Add Dropout / BatchNorm', 'More data augmentation', 'L2 weight decay'],
    },
    {
        'symptom':  'Train loss high, val loss high (underfitting)',
        'causes':   ['Model too small', 'LR too low', 'Not enough epochs'],
        'fixes':    ['Larger model', 'Increase LR', 'Train longer', 'Check preprocessing is correct'],
    },
    {
        'symptom':  'Loss oscillates, never converges',
        'causes':   ['LR too high', 'Batch size too small', 'Missing gradient clipping'],
        'fixes':    ['Lower LR', 'Larger batch or gradient accumulation', 'clip_grad_norm_(model.parameters(), 1.0)'],
    },
    {
        'symptom':  'GPU out of memory (OOM)',
        'causes':   ['Batch size too large', 'Model too large', 'Accumulating graphs'],
        'fixes':    ['Reduce batch_size', 'Use gradient checkpointing', 'Call .detach() on intermediate tensors'],
    },
]

for check in checks:
    print(f"  Symptom: {check['symptom']}")
    print(f"  Causes:")
    for c in check['causes']: print(f"    - {c}")
    print(f"  Fixes:")
    for f in check['fixes']:  print(f"    ✓ {f}")
    print()

## Interview Questions & Answers

---

**Q1: What is the vanishing gradient problem and how does BatchNorm address it?**

A: During backpropagation, gradients are multiplied layer by layer. With many layers and activations like sigmoid (which saturate near 0/1), gradients shrink exponentially — early layers get near-zero updates and never learn. **Batch Normalisation** normalises layer inputs to zero mean, unit variance within each mini-batch, keeping activations in the linear regime of activation functions where gradients are non-zero. It also adds learnable scale (γ) and shift (β) so the network can undo the normalisation if needed. This allows training networks 10-100× deeper reliably.

---

**Q2: Why do we use CrossEntropyLoss instead of MSELoss for classification?**

A: MSE penalises wrong predictions quadratically regardless of confidence. CrossEntropy = -log(p_correct), which penalises **confident wrong predictions extremely harshly** (log(0.01) = -4.6) and barely penalises uncertain wrong predictions. This matches the probabilistic interpretation — the model learns proper probability distributions. MSELoss on class labels treats class 0 and class 2 as having a numeric distance relationship that doesn't exist. Additionally, CrossEntropy combined with softmax has a clean gradient: `softmax(logits) - one_hot(label)`, making training stable.

---

**Q3: What is gradient clipping and when is it essential?**

A: Gradient clipping caps the L2 norm of the gradient vector at `max_norm`. When a batch produces an unusually large loss (outlier data, early training instability), the gradient can be enormous — the parameter update overshoots a good minimum and the loss spikes to NaN, crashing training. Clipping rescales the gradient if it exceeds the cap without changing its direction. Essential for: RNNs/LSTMs (inherently unstable gradients), early training epochs (random weights produce chaotic gradients), and any transformer architecture. `max_norm=1.0` is a safe default.

---

**Q4: What is early stopping and why do we save the best checkpoint rather than train for fixed epochs?**

A: As training continues past the optimum, the model overfits — training loss falls but validation loss rises. Early stopping halts training when validation loss hasn't improved for `patience` epochs, saving compute. We checkpoint the weights at the *lowest validation loss*, not the last epoch, because: the last epoch is more overfit than the best; re-running training to find the right epoch count wastes GPU time; different runs may reach optimum at different epochs. Best practice: checkpoint every epoch if validation loss improved, then load the best weights for final evaluation.

---

**Q5: What is the difference between `torch.jit.script` and `torch.jit.trace` for deployment?**

A: Both convert PyTorch to TorchScript (runs without Python runtime — needed for C++, mobile, edge devices). **`script`** parses Python source code and preserves all control flow (if/for/while). Works on any model but fails if you use unsupported Python syntax. **`trace`** records a single forward pass — control flow is baked in (if `batch_size != 1` changes behaviour, trace will always produce the batch_size=1 path). Use `script` for models with data-dependent shapes or branches; `trace` for simple feed-forward models. When deploying, prefer ONNX for cross-framework interoperability.

---

**Q6: How do you debug a neural network that outputs the same class for every input?**

A: This classic failure mode ("collapsed predictions") has several causes: (1) **Learning rate too high** — weights explode, all logits become equal after softmax; fix: lower LR. (2) **Class imbalance** — model learns to always predict majority class; fix: `class_weight`, weighted sampler, or focal loss. (3) **Dead ReLUs** — neurons stuck at 0; fix: use LeakyReLU or check initialisation. (4) **Wrong loss** — e.g., using MSE for multi-class; fix: CrossEntropyLoss. (5) **Bug in data pipeline** — labels not aligned with features; fix: visualise `(image, label)` pairs before training.

## Recommended Resources

| Resource | Link | Why |
|---|---|---|
| PyTorch Tutorials | https://pytorch.org/tutorials/ | Official deep learning guide |
| fast.ai Course | https://course.fast.ai/ | Best practical DL course (free) |
| BatchNorm Paper | https://arxiv.org/abs/1502.03167 | Ioffe & Szegedy 2015 |
| AdamW Paper | https://arxiv.org/abs/1711.05101 | Decoupled weight decay |
| OneCycleLR Paper | https://arxiv.org/abs/1708.07120 | Superconvergence |
| ONNX | https://onnx.ai/ | Cross-platform model export |


## Summary: Production DL Checklist

```
✅ Data: torchvision transforms for augmentation, proper normalization
✅ Model: CNN with BatchNorm + Dropout; Kaiming init for conv, Xavier for linear
✅ Loss: CrossEntropyLoss with label_smoothing=0.1 (prevents overconfidence)
✅ Optimizer: AdamW (Adam + decoupled weight decay) → generalization
✅ Scheduler: OneCycleLR — fast convergence, good regularization
✅ Gradient clipping: clip_grad_norm_(model.parameters(), max_norm=1.0)
✅ Early stopping with checkpoint: save best model, stop when no improvement
✅ Per-class accuracy: identify which classes are hardest
✅ Export: TorchScript for Python-free production serving
✅ API: FastAPI endpoint with image upload
```

### Key Lessons
1. **BatchNorm + Dropout together**: BatchNorm on conv layers, Dropout on FC layers
2. **Always validate per epoch**: training accuracy alone is misleading
3. **Save the full checkpoint**: model state + optimizer state + epoch (for resuming)
4. **script() over trace()**: TorchScript with `script()` handles control flow; `trace()` just replays one path
5. **Business metric matters**: per-class accuracy reveals which digits are hard

### Extensions
- **Transfer learning**: use pretrained ResNet backbone (replace final layer)
- **Mixed precision**: `torch.cuda.amp.autocast()` for 2× speedup on GPU
- **Distributed training**: `torch.nn.DataParallel` or `DistributedDataParallel`